# Time Series Modelling and Eyetracking Data

In [ ]:
set.seed(42)

n <- 25
phi <- 0.5       # AR(1) coefficient
theta <- 0.5     # MA(1) coefficient
sd_innov <- 0.5  # innovation SD
mu <- 10         # positive starting level

# simulate stationary ARMA(1,1) component
z <- as.numeric(arima.sim(n = n, model = list(ar = phi, ma = theta), sd = sd_innov))

# start at mu so the series remains centered around a positive level and add drift

raw_dat <- mu + z + (seq(1,n,1)*0.2)
# cumsum(z)
raw_dat

In [ ]:
plot(density(exp(raw_dat)))

In [ ]:
plot(raw_dat, type = "l", col = "grey40", ylab = "counts",
     main = "Simulated Log Transformed ET data")

In [ ]:
dat=diff(raw_dat)

In [ ]:
plot(dat, type = "l", col = "grey40", ylab = "counts",
     main = "Simulated Log Transformed ET data")

In [ ]:
drift_rate=0.2
drift=seq(1,length(raw_dat),1)*drift_rate
dat=raw_dat-drift

In [ ]:
plot(dat, type = "l", col = "grey40", ylab = "counts",
     main = "Simulated Log Transformed ET data")

In [ ]:
fit1 <- arima(dat, order = c(0, 0, 1), method = "ML")

In [ ]:
# R
# extract coefficients from fitted model
theta1 <- coef(fit1)["ma1"]    # MA(1) coefficient
intercept <- coef(fit1)["intercept"]  # mean/intercept term

# MA(1): y_t = mu + e_t + theta1 * e_{t-1}
# one-step-ahead fitted value = mu + theta1 * e_{t-1}
# where e_t = y_t - fitted_t (recursively)

n_obs <- length(dat)
resids <- numeric(n_obs)
fitted_manual <- numeric(n_obs)

# initialise: e_0 = 0
resids[1] <- dat[1] - intercept
fitted_manual[1] <- intercept

for (t in 2:n_obs) {
  fitted_manual[t] <- intercept + theta1 * resids[t - 1]
  resids[t] <- dat[t] - fitted_manual[t]
}

# compare manual residuals to arima residuals
cor(resids, as.numeric(residuals(fit1)))   # should be ~1

# plot
plot(dat, type = "l", col = "grey40", ylab = "counts",
     main = "MA(1): observed vs one-step-ahead predicted")
lines(fitted_manual, col = "firebrick", lwd = 1.5)
legend("topright", legend = c("observed", "fitted"),
       col = c("grey40", "firebrick"), lty = 1)

In [ ]:
# fit AR(1) model
fit1 <- arima(dat, order = c(1,0,0))

# extract coefficients
phi1 <- coef(fit1)["ar1"]
intercept <- coef(fit1)["intercept"]

n_obs <- length(dat)
resids <- numeric(n_obs)
fitted_manual <- numeric(n_obs)

# initialize
fitted_manual[1] <- intercept
resids[1] <- dat[1] - fitted_manual[1]

# AR(1) recursion:
# y_hat_t = mu + phi * (y_{t-1} - mu)
for (t in 2:n_obs) {
  fitted_manual[t] <- intercept + phi1 * (dat[t - 1] - intercept)
  resids[t] <- dat[t] - fitted_manual[t]
}

# compare manual residuals to model residuals
cor(resids, as.numeric(residuals(fit1)), use = "complete.obs")

# plot
plot(dat, type = "l", col = "grey40", ylab = "value",
     main = "AR(1): observed vs one-step-ahead predicted")

lines(fitted_manual, col = "firebrick", lwd = 1.5)

legend("topright",
       legend = c("observed", "fitted"),
       col = c("grey40", "firebrick"),
       lty = 1,
       lwd = c(1, 1.5))

In [ ]:
set.seed(42)

n <- length(raw_dat)
t <- 1:n

fit1 <- arima(raw_dat,
              order = c(1,0,0),
              xreg = t,
              include.mean = TRUE)

# coefficients
alpha <- coef(fit1)["intercept"]
beta  <- coef(fit1)["t"]
phi1  <- coef(fit1)["ar1"]

# storage
fitted_vals <- numeric(n)
resids <- numeric(n)

# initialize
fitted_vals[1] <- alpha + beta * t[1]
resids[1] <- raw_dat[1] - fitted_vals[1]

# recursive AR(1) with deterministic trend
for (i in 2:n) {
  mean_t <- alpha + beta * t[i]

  fitted_vals[i] <- mean_t +
    phi1 * (raw_dat[i - 1] - (alpha + beta * t[i - 1]))

  resids[i] <- raw_dat[i] - fitted_vals[i]
}

# plot
plot(raw_dat, type = "l", col = "grey40",
     main = "AR(1) with linear drift (manual fitted values)")

lines(fitted_vals, col = "firebrick", lwd = 2)

# CELER data import

In [ ]:
library(tidyverse)

In [ ]:
url="https://www.dropbox.com/scl/fi/nsknopddl7n6pcvd4q4ne/languages.tsv?rlkey=64k6goa4e7jc86dbokyp0if1g&st=kl2e4ely&dl=1"
download.file(url, "languages.tsv")
url="https://www.dropbox.com/scl/fi/u1om6hdc09v2ubeita8ge/metadata.tsv?rlkey=h818h6nh8l19wq2x98o0cv87i&st=su3nanyn&dl=1"
download.file(url, "metadata.tsv")
url="https://www.dropbox.com/scl/fi/doeyekcm8fvw9edgzp24m/sent_fix.tsv?rlkey=eftwbmkk8t44jsfxpiinyr4tc&st=ygd8564c&dl=1"
download.file(url, "sent_fix.tsv")
url="https://www.dropbox.com/scl/fi/zodjixb75wvvszzx3i87w/sent_ia.tsv?rlkey=m46r93dpuvdl44v94cwoix07e&st=e6zw11qq&dl=1"
download.file(url, "sent_ia.tsv")


### Load data and metadata

In [ ]:
dat_fix=read.delim("sent_fix.tsv",quote="")
dat_ia=read.delim("sent_ia.tsv",quote="")
languages=read.delim("languages.tsv",quote="")
metadata=read.delim("metadata.tsv",quote="")


### Merge data and metadata

In [ ]:
names(languages)[1] <- tolower(names(languages)[1])
names(metadata)[1] <- tolower(names(metadata)[1])
dat_fix=merge(dat_fix,languages,by="list",all.x=TRUE)
dat_fix=merge(dat_fix,metadata,by="list",all.x=TRUE)
dat_ia=merge(dat_ia,languages,by="list",all.x=TRUE)
dat_ia=merge(dat_ia,metadata,by="list",all.x=TRUE)